# cycle-detection-temp-set — worked example 1: Detect a cycle in a dict-of-lists adjacency graph

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Cycle detection by DFS uses **two** sets, not one. `perm` marks vertices whose entire subtree is *finished* — re-seeing one is a legal shared descendant in a DAG. `temp` (the gray set) marks vertices *currently on the recursion stack*; re-seeing one of those is a back-edge, i.e. a cycle. Naive single-`visited`-set detection false-positives on diamonds because it can't tell 'finished' from 'still in progress'.

## Worked solution

We are given a graph as `adj: dict[str, list[str]]` and must return `True` iff any cycle is reachable from a given start node.

**Step 1 — two sets.** `perm` holds finished vertices, `temp` holds vertices on the active DFS path. We key on the node string itself here (hashable), not `id()`, because the nodes are plain strings.

**Step 2 — the gray check comes BEFORE adding.** Inside `visit(u)`, first return `False` if `u in perm` (its subtree is proven acyclic, skip it). Then return `True` if `u in temp` — we have re-entered a vertex still on the stack, which can only happen via a back-edge, so a cycle exists.

**Step 3 — color gray, recurse, color black.** Add `u` to `temp` (gray) before exploring, recurse into every neighbour, and if any recursion reports a cycle propagate `True` immediately.

**Step 4 — un-gray on the way out.** This is the step novices forget. After the loop, `temp.remove(u)` then `perm.add(u)`. Removing from `temp` is essential: once we finish `u`'s subtree, `u` is no longer on the stack, so a *sibling* path reaching `u` must NOT be flagged — it's a shared descendant, not a cycle.

**Why it's correct.** A directed graph has a cycle iff a DFS encounters a back-edge — an edge to a vertex still on the recursion stack. The `temp` set is exactly 'on the stack', so membership in `temp` at visit time is precisely the back-edge condition.

In [ ]:
def has_cycle(adj, start):
    perm, temp = set(), set()

    def visit(u):
        if u in perm:
            return False        # finished subtree -- legal
        if u in temp:
            return True         # back-edge -- cycle
        temp.add(u)
        for v in adj.get(u, []):
            if visit(v):
                return True
        temp.remove(u)          # leaving the stack
        perm.add(u)
        return False

    return visit(start)


# Diamond DAG: a->b, a->c, b->d, c->d  (NOT a cycle)
dag = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
# Cyclic: a->b->c->a
cyc = {"a": ["b"], "b": ["c"], "c": ["a"]}
print("diamond DAG has_cycle:", has_cycle(dag, "a"))
print("a->b->c->a has_cycle:", has_cycle(cyc, "a"))